# Crop Disease Inference Server (Colab)
Run the FastAPI server on Colab GPU and expose it via ngrok.

**Prerequisites (on your Google Drive):**
- `MyDrive/CropDisease/weights/sam2.1_t.pt`
- `MyDrive/CropDisease/weights/best_classifier.pth`
- `MyDrive/CropDisease/data/TextEmbeddings/*.pt`
- The cloned project repo (or upload manually). This notebook assumes the repo will be cloned in `/content`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the project repo
Replace the URL with your repo. Skip if you upload the code manually instead.

In [ ]:
%cd /content
!git clone https://github.com/<YOUR_USER>/Fast-and-Robust-Crop-Disease-Prediction.git project
%cd /content/project

## 3. Link model files from Drive into the project tree
Server reads from `weights/` and `data/TextEmbeddings/` (overridable via env vars).

In [ ]:
import os, pathlib
DRIVE_ROOT = '/content/drive/MyDrive/CropDisease'
os.makedirs('weights', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Symlink weights and text embeddings (overwrite if exist)
for src, dst in [
    (f'{DRIVE_ROOT}/weights/sam2.1_t.pt', 'weights/sam2.1_t.pt'),
    (f'{DRIVE_ROOT}/weights/best_classifier.pth', 'weights/best_classifier.pth'),
    (f'{DRIVE_ROOT}/data/TextEmbeddings', 'data/TextEmbeddings'),
]:
    if os.path.lexists(dst):
        os.remove(dst) if not os.path.isdir(dst) or os.path.islink(dst) else None
    os.symlink(src, dst)
    print('linked', dst, '->', src)

!ls -la weights data/TextEmbeddings | head

## 4. Install dependencies

In [ ]:
# Colab already has torch/torchvision/opencv. Install only the rest.
!pip install -q -r server/requirements.txt

## 5. Configure ngrok
Get a free authtoken from https://dashboard.ngrok.com/get-started/your-authtoken and paste it below.

In [ ]:
NGROK_AUTHTOKEN = ''  # <-- paste your ngrok authtoken here
from pyngrok import ngrok, conf
if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()  # clear any leftover tunnels
public_url = ngrok.connect(8000, 'http')
print('Public URL:', public_url)

## 6. Launch the FastAPI server (background)
Uvicorn runs in the background; logs are tailed in the next cell.

In [ ]:
import subprocess, time, os
os.environ['PYTHONPATH'] = '/content/project'
log = open('/content/server.log', 'w')
proc = subprocess.Popen(
    ['uvicorn', 'server.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log, stderr=subprocess.STDOUT, cwd='/content/project'
)
print('uvicorn PID:', proc.pid)
time.sleep(3)
!tail -n 40 /content/server.log

## 7. Smoke test

In [ ]:
import requests
print(requests.get('http://127.0.0.1:8000/health').json())
print(requests.get('http://127.0.0.1:8000/crops').json())

## 8. Sample /predict call
Replace `sample.jpg`, crop name, and bbox with your inputs.

In [ ]:
import requests
files = {'image': open('sample.jpg', 'rb')}
data = {'crop': 'cucumber', 'bbox': '50,50,400,400', 'return_masked_image': 'false'}
r = requests.post('http://127.0.0.1:8000/predict', files=files, data=data, timeout=120)
print(r.status_code)
print(r.json())

## 9. (Optional) Stop the server

In [ ]:
proc.terminate(); proc.wait(); ngrok.kill()
print('stopped')